# 00 · The question, available data, and the first decision

**Can we remove tracker errors without deleting movements that really
happened?** This notebook establishes what data and public-model inputs
are available. The first scientific result is a preservation-versus-repair
comparison, not binary gait classification.

Run each notebook in a fresh kernel, in order **00 → 04**. Notebook 05 is
an external visual stress test. These are offline restoration experiments:
the model may inspect the declared complete clip. They do not claim causal
forecasting or clinical diagnosis.

**The default is real data.** Export `MP_RUN_ROOT`, the AMASS and GAVD data
paths, and the model configuration before opening Jupyter. See the
[launch guide](../../slurm/motion-preservation/README.md).
For a CPU walkthrough of the mechanics, explicitly choose `MP_MODE=demo`
and a separate run directory. Demo outputs cannot establish a research result.

[Proposal](../../docs/studies/motion-preservation/protocol/proposal.md)
· [Notebook guide](README.md)

In [ ]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

project_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(project_override).expanduser()] if project_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)  # Resolve manifest/config paths from the checkout in every kernel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, Video, display
from gavd6_sjepa.research_directions.motion_preservation import workflow, reporting

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
cfg = workflow.config_from_environment()
RUN_ROOT = Path(cfg.run_root)
print(f"Mode: {cfg.mode}; run directory: {RUN_ROOT}")
if cfg.mode == "demo":
    display(Markdown("**DEMO ONLY: generated fixtures and stand-in models. "
                     "These outputs are not evidence about AMASS, GAVD, or a pretrained prior.**"))

## 1. Why ordinary reconstruction error is not enough

Suppose a real ankle excursion lasts only a few frames. Removing that
excursion may slightly reduce the average error across the entire body,
even though the useful movement information was lost.

We need two separate measurements:

- **Repair:** how much injected tracking error is removed?
- **Preservation:** how much of the known real movement remains?

We compare methods at a repair strength chosen on separate calibration
people. A method cannot win by leaving every noisy coordinate unchanged.

In [ ]:
# This table explains the design; it contains no measured results.
display(pd.DataFrame([
    {"True event": False, "Tracking noise": False, "Desired behavior": "Keep the clean movement"},
    {"True event": True,  "Tracking noise": False, "Desired behavior": "Keep the event"},
    {"True event": False, "Tracking noise": True,  "Desired behavior": "Remove the error"},
    {"True event": True,  "Tracking noise": True,  "Desired behavior": "Preserve the event and remove the error"},
]))

## 2. Read the existing AMASS and GAVD manifests

AMASS supplies known body motion for controlled experiments. We use
whole-body joints so arms and trunk can carry a true event. GAVD supplies
in-the-wild videos for external inspection. Its estimated trajectories
are not 3D reference truth.

Inventory counts come from the repository manifests. Availability comes
from the configured filesystem on this machine. A manifest entry does
not imply that its raw file is available locally.

In [ ]:
started = perf_counter()
inventory = workflow.inventory(cfg)
for name in ("amass", "gavd", "availability"):
    table = inventory[name]
    display(Markdown(f"### {name.capitalize()} ({len(table):,} rows)"))
    display(table.head(12))
print(f"Inventory completed in {perf_counter() - started:.1f} seconds.")

In [ ]:
# Dataset/identity summaries are descriptive, not an eligibility decision.
amass = inventory["amass"]
for column in ("source_dataset", "split", "role"):
    if column in amass:
        display(amass.groupby(column, dropna=False).size().rename("manifest_rows").to_frame())
identity_column = next((c for c in ("person_id", "identity", "subject_id_candidate", "audited_subject_id")
                        if c in amass), None)
if identity_column:
    print(f"AMASS identities/groups represented: {amass[identity_column].nunique():,}")
gavd = inventory["gavd"]
if "video_id" in gavd:
    print(f"GAVD source recordings represented: {gavd.video_id.nunique():,}")

## 3. Keep development and final evaluation separate

People are assigned before generating event or corruption variants. All
variants of a person stay together. Train the gate on an arm-leg timing
event. Use a foot-clearance event on development people for the first
decision. Once that result affects our choices, it is development data.
The trunk-pelvis event on final people remains unopened until notebook
04 is explicitly launched in final mode.

The current final configuration changes the event family, camera angle
and corruption mechanism together. This is a combined stress test; it
does not isolate which individual change caused a success or failure.

Audited AMASS identity determines the grouping when available. Uncertain
identities support only a weaker group-level claim. A new adaptation
split does not establish absence from a public prior's training data.

In [ ]:
display(pd.DataFrame([
    ("train", "Learn the small gate", "Development event and corruption settings"),
    ("calibration", "Choose strength and decision thresholds", "Separate people; no gradient fitting"),
    ("development", "48-hour continue/stop decision", "Held people and development event family"),
    ("final", "One final held-event evaluation", "Reserved people, event, and configured nuisance condition"),
], columns=["Role", "Purpose", "Boundary"]))

## 4. Read the available backend configuration

A real run needs a successfully loaded released motion prior and an
image-motion estimate. The model bridge must state its representation,
coordinates, frame rate and inverse conversion. The quick synthetic
fixture has a different purpose and is always labelled demo.

Missing model weights or body-model assets are practical blockers. They
must not silently select a smoothing model and call it a pretrained prior.

In [ ]:
config_path = RUN_ROOT / "config.json"
if config_path.is_file():
    saved_config = json.loads(config_path.read_text())
    display(pd.DataFrame([{"setting": key, "value": str(value)}
                          for key, value in saved_config.items()]))
else:
    print("Configuration is available in cfg; the workflow has not written config.json yet.")

## Decision before spending GPU time

Continue when the selected AMASS files, body assets, prior and flow
backend are available and there are enough distinct people to separate
fitting from evaluation. First use a small pilot. Measure actual elapsed
time before scaling to the proposal's planned 128 motion instances.

Next: [01 · Make controlled pairs](01_make_controlled_pairs.ipynb).